# PMAG (Protein Metal Affinity GCN)

This project is aimed at calculating the metal affinity for proteins in various bacteria. It uses the BindingDB database and a Graph Convolutional Network. Each molecule is expressed as a graph where the edges are molecules bonded to each other and each data point is a molecule.

## Processing Data

The .sdf file first has to be turned into a graph with the features, where the graph is a torch_geometric.data.Data object, containing the node and edge tables. The node features are

- element = a one-hot encoding of the element
- hybridization = the angle & shape of the bond the atom prefers
- charge = the formal charge of the atom

The edge features are

- bond type = how strong the bond is
- aromaticity = whether the electrons can change between the atoms

In [ ]:
import torch_geometric
# Calculating a feature - replace GetHybridization() with whatever
from rdkit import Chem

def grab_first_valid(mols):
    for mol in mols:
        if mol != None: return mol

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)
print(list(supplier[0].GetPropNames()))
print('Loaded')

mol = grab_first_valid(supplier)
Chem.SanitizeMol(mol)

for atom in mol.GetAtoms():
    print(atom.GetHybridization())

In [ ]:
# Generating one-hot encodings
from rdkit import Chem
from joblib import dump

path_mols = 'data/train_input.csv'
supplier = Chem.SmilesMolSupplier(path_mols, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

ids = {}

for mol in supplier:
    if mol == None: continue

    for atom in mol.GetAtoms():
        if atom.GetSymbol() in ids:
            pass
        else:
            ids[atom.GetSymbol()] = len(ids) + 1

encodings = {}

for i, id in enumerate(ids.items()):
    vector_encoded = [0] * len(ids)
    vector_encoded[i] = 1
    encodings[id[0]] = vector_encoded

dump(encodings, 'data/one-hot.joblib')

In [ ]:
# Graph creation code - run workers.py instead because of IPython multiprocessing issues
import numpy as np
from torch_geometric.data import Data
import multiprocessing as mp
from rdkit import Chem
from joblib import load

chunk_size = 200

def get_chunks(file_path, chunk_size):
    chunk = []
    supplier = Chem.SmilesMolSupplier(file_path, delimiter=',', smilesColumn=0, nameColumn=-1, titleLine=True)

    for mol in supplier:
        if mol is None: continue
        chunk.append(mol)

        if len(chunk) == chunk_size:
            yield chunk
            chunk = []

encodings = load('./data/one-hot.joblib')

def process(chunk):
    chunk_to_save = []
    for mol in chunk:
        X = []
        edge_features = []

        edge_matrix_idx1 = []
        edge_matrix_idx2 = []

        for atom in mol.GetAtoms():
            X.append([*encodings[atom.GetSymbol()], int(atom.GetHybridization()), atom.GetFormalCharge()])

        for bond in mol.GetBonds():
            features_for_edge = [bond.GetBondType().real, float(bond.GetIsAromatic())]
            edge_features.append(features_for_edge)
            edge_features.append(features_for_edge)

            edge_matrix_idx1.append(bond.GetBeginAtomIdx())
            edge_matrix_idx2.append(bond.GetEndAtomIdx())

            edge_matrix_idx2.append(bond.GetBeginAtomIdx())
            edge_matrix_idx1.append(bond.GetEndAtomIdx())

        edge_matrix = torch.from_numpy(np.array([edge_matrix_idx1, edge_matrix_idx2]))
        X = torch.from_numpy(np.array(X))
        edge_features = torch.from_numpy(np.array(edge_features))
        graph = Data(x=X, edge_index=edge_matrix, edge_attr=edge_features)

        chunk_to_save.append(graph)
        if len(chunk_to_save) == chunk_size:
            if len(os.listdir('data/graph_chunks')) > 0:
                next_to_save = max([int(filename[:4]) + 1 for filename in os.listdir('data/graph_chunks')])
            else:
                next_to_save = 1
            torch.save(chunk_to_save, f'./data/graph_chunks/{next_to_save:04d}.pt')

if __name__ == '__main__':
    mols_path = 'data/train_input.csv'

    with mp.Pool(processes=5) as pool:
        results = pool.imap(process, get_chunks(mols_path, chunk_size))
        for _ in results: pass

## Training the Model

The model is a graph convolutional network with message passing in pytorch.

In [ ]:
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import lightning as L
import torch, gc
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_add_pool
import torch_geometric

class PMAG(L.LightningModule):
    def __init__(self, node_dim, edge_dim):
        super().__init__()

        layers = []

        for part in range(4):
            if part == 0:
                layers.append(nn.Linear(node_dim, 128))
            else:
                layers.append(nn.Linear(128, 128))

            layers.append(nn.LayerNorm(128))
            layers.append(nn.ReLU())

            layers.append(nn.Linear(128, 128))
            layers.append(nn.LayerNorm(128))
            layers.append(nn.ReLU())

        mlp = nn.Sequential(*layers)
        self.conv = GINEConv(nn=mlp, train_eps=True, edge_dim=edge_dim)
        self.head = nn.Linear(128, 1)

    def forward(self, data):
        X = F.relu(self.conv(data.x, data.edge_index, data.edge_attr))
        embeds = global_add_pool(X, data.batch)
        return self.head(embeds)

    def training_step(self, batch, batch_idx):
        pred = self(batch).squeeze(-1)
        loss = F.mse_loss(pred, batch.y)
        self.log("train_loss", loss.item(), prog_bar=True, on_step=True, on_epoch=True, batch_size=batch.num_graphs)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

    def on_epoch_end(self):
        loss = self.trainer.callback_metrics.get("train_loss")
        print(f'Epoch {self.current_epoch}: train loss: {loss:.4f}')

        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

graphs = []

for filepath in os.listdir('./data/graph_chunks'):
    if filepath.endswith('.pt'):
        chunk = torch.load(os.path.join('./data/graph_chunks', filepath), weights_only=False)

        for graph in chunk:
            graph.x = graph.x.float()
            graph.edge_attr = graph.edge_attr.float()
            graph.edge_index = graph.edge_index.long()

            try:
                if isinstance(graph.y, torch.Tensor):
                    pass
                else:
                    graph.y = torch.tensor(float(graph.y), dtype=torch.float32)
            except ValueError:
                chunk.remove(graph)

            graphs.extend(chunk)

batch_size = 32
t_loader = torch_geometric.loader.DataLoader(graphs, batch_size=batch_size, num_workers=5)

checkpoint = L.pytorch.callbacks.ModelCheckpoint(
    dirpath='./models',
    filename='./models/{epoch:02d}-{train_loss:.2f}',
    monitor='train_loss',
    mode='min',
)

model = PMAG(graphs[0].x.shape[1], graphs[0].edge_attr.shape[1])
trainer = L.Trainer(max_epochs=10, callbacks=[checkpoint], accelerator='mps', devices=1)
trainer.fit(model=model, train_dataloaders=t_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv │ GINEConv │  128 K │ train │     0 │
│ 1 │ head │ Linear   │    129 │ train │     0 │
└───┴──────┴──────────┴────────┴───────┴───────┘

Trainable params: 129 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 129 K                                                                                                
Total estimated model params size (MB): 0.516                                                                      
Modules in train mode: 29                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/torch_geometric/data/collate.py:203: UserWarning: An output with one or more elements was resized since it had shape [876], which does not match the required output shape [876, 2]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:37.)
  value = torch.cat(values, dim=cat_dim or 0, out=out)
/Users/zacharypotishko/Desktop/BiochemistryRAG/.venv/lib/python3.11/site-packages/torch_geometric/data/co